# 07 · Model Comparison

**Project:** Enterprise HR AI  
**Sources:** `data/processed/features_unscaled.csv` & `data/processed/features_scaled.csv`  
**Task:** Evaluate Logistic Regression, Random Forest, and XGBoost across both unscaled and scaled datasets using the identical 80/20 stratified split (`random_state=42`). Re-apply strict Recall (primary) and F1 (secondary) decision rule across all 6 models.

---

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

PROC = os.path.join('..', 'data', 'processed')
MODELS = os.path.join('..', 'models')
os.makedirs(MODELS, exist_ok=True)

RANDOM_STATE = 42
print('PROC  :', os.path.abspath(PROC))
print('MODELS:', os.path.abspath(MODELS))

PROC  : C:\Users\ASUS\Desktop\enterprise_hr_ai\data\processed
MODELS: C:\Users\ASUS\Desktop\enterprise_hr_ai\models


---
## Step 1 · Load & Split Unscaled Data

Loading `features_unscaled.csv` with 80/20 stratified split (`random_state=42`).

In [2]:
df_unscaled = pd.read_csv(os.path.join(PROC, 'features_unscaled.csv'))
print(f'Loaded features_unscaled.csv: {df_unscaled.shape[0]:,} rows x {df_unscaled.shape[1]} cols')

X_unscaled = df_unscaled.drop(columns=['Attrition'])
y_unscaled = df_unscaled['Attrition']

X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X_unscaled, y_unscaled, test_size=0.20, stratify=y_unscaled, random_state=RANDOM_STATE
)

test_attrition_pct = y_test_u.mean() * 100
print(f'Unscaled Train shape: {X_train_u.shape}, Test shape: {X_test_u.shape}')
print(f'Test attrition rate: {test_attrition_pct:.2f}% ({y_test_u.sum()} / {len(y_test_u)})')
assert abs(test_attrition_pct - 15.99) < 0.05, f'Test class balance mismatch: got {test_attrition_pct:.2f}%'
print('CONFIRMED: Split reproduces notebook 06 class balance (15.99%).')

Loaded features_unscaled.csv: 1,470 rows x 49 cols
Unscaled Train shape: (1176, 48), Test shape: (294, 48)
Test attrition rate: 15.99% (47 / 294)
CONFIRMED: Split reproduces notebook 06 class balance (15.99%).


---
## Step 2 · Train Models on Unscaled Features

1. **Logistic Regression (Unscaled)** (`max_iter=1000`, `random_state=42`)
2. **Random Forest (Unscaled)** (`n_estimators=200`, `random_state=42`)
3. **XGBoost (Unscaled)** (`random_state=42`, `eval_metric='logloss'`)

In [3]:
models_unscaled = {}

print('Training Logistic Regression (unscaled)...')
lr_u = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_u.fit(X_train_u, y_train_u)
models_unscaled['Logistic Regression (Unscaled)'] = lr_u

print('Training Random Forest (unscaled)...')
rf_u = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf_u.fit(X_train_u, y_train_u)
models_unscaled['Random Forest (Unscaled)'] = rf_u

print('Training XGBoost (unscaled)...')
xgb_u = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', n_estimators=100)
xgb_u.fit(X_train_u, y_train_u)
models_unscaled['XGBoost (Unscaled)'] = xgb_u

print('Unscaled models trained successfully.')

Training Logistic Regression (unscaled)...


Training Random Forest (unscaled)...


Training XGBoost (unscaled)...
Unscaled models trained successfully.


---
## Step 3 · Train Models on Scaled Features

Loading `features_scaled.csv` with identical 80/20 stratified split (`random_state=42`).
Evaluating Random Forest and XGBoost on scaled features.

In [4]:
df_scaled = pd.read_csv(os.path.join(PROC, 'features_scaled.csv'))
print(f'Loaded features_scaled.csv: {df_scaled.shape[0]:,} rows x {df_scaled.shape[1]} cols')

X_scaled = df_scaled.drop(columns=['Attrition'])
y_scaled = df_scaled['Attrition']

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_scaled, y_scaled, test_size=0.20, stratify=y_scaled, random_state=RANDOM_STATE
)

models_scaled = {}

print('Training Random Forest (scaled, n_estimators=200)...')
rf_s = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf_s.fit(X_train_s, y_train_s)
models_scaled['Random Forest (Scaled)'] = rf_s

print('Training XGBoost (scaled, eval_metric=logloss)...')
xgb_s = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', n_estimators=100)
xgb_s.fit(X_train_s, y_train_s)
models_scaled['XGBoost (Scaled)'] = xgb_s

print('Scaled tree models trained successfully.')

Loaded features_scaled.csv: 1,470 rows x 49 cols


Training Random Forest (scaled, n_estimators=200)...


Training XGBoost (scaled, eval_metric=logloss)...
Scaled tree models trained successfully.


---
## Step 4 · Complete 6-Row Comparison Table

Metrics computed on the test set (`support = 47` leavers, `247` non-leavers).

In [5]:
results = []

# 1. Baseline LogReg (Scaled - Step 6)
results.append({
    'Model': 'Baseline LogReg (Scaled - Step 6)',
    'Precision': 0.6538,
    'Recall': 0.3617,
    'F1': 0.4658,
    'ROC-AUC': 0.8134
})

# 2. Unscaled models
for name, model in models_unscaled.items():
    y_pred = model.predict(X_test_u)
    y_prob = model.predict_proba(X_test_u)[:, 1]
    results.append({
        'Model': name,
        'Precision': precision_score(y_test_u, y_pred, zero_division=0),
        'Recall': recall_score(y_test_u, y_pred, zero_division=0),
        'F1': f1_score(y_test_u, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test_u, y_prob)
    })

# 3. Scaled tree models
for name, model in models_scaled.items():
    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)[:, 1]
    results.append({
        'Model': name,
        'Precision': precision_score(y_test_s, y_pred, zero_division=0),
        'Recall': recall_score(y_test_s, y_pred, zero_division=0),
        'F1': f1_score(y_test_s, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test_s, y_prob)
    })

comparison_df = pd.DataFrame(results).set_index('Model')
print('=== COMPLETE 6-ROW MODEL COMPARISON TABLE ===')
print(comparison_df.to_string())

=== COMPLETE 6-ROW MODEL COMPARISON TABLE ===
                                   Precision  Recall     F1  ROC-AUC
Model                                                               
Baseline LogReg (Scaled - Step 6)     0.6538  0.3617 0.4658   0.8134
Logistic Regression (Unscaled)        0.7857  0.2340 0.3607   0.7561
Random Forest (Unscaled)              0.5556  0.1064 0.1786   0.7930
XGBoost (Unscaled)                    0.5652  0.2766 0.3714   0.7735
Random Forest (Scaled)                0.5556  0.1064 0.1786   0.7913
XGBoost (Scaled)                      0.5652  0.2766 0.3714   0.7735


In [6]:
print('=== DETAILED CONFUSION MATRICES ===\n')
for name, model in models_unscaled.items():
    cm = confusion_matrix(y_test_u, model.predict(X_test_u))
    print(f'{name}: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')
for name, model in models_scaled.items():
    cm = confusion_matrix(y_test_s, model.predict(X_test_s))
    print(f'{name}: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

=== DETAILED CONFUSION MATRICES ===

Logistic Regression (Unscaled): TN=244, FP=3, FN=36, TP=11
Random Forest (Unscaled): TN=243, FP=4, FN=42, TP=5
XGBoost (Unscaled): TN=237, FP=10, FN=34, TP=13
Random Forest (Scaled): TN=243, FP=4, FN=42, TP=5
XGBoost (Scaled): TN=237, FP=10, FN=34, TP=13


---
## Step 5 · Honest Decision Rule Application

**Decision Rules:**
- Primary criterion: **Recall** (missing a leaver is the costly error)
- Secondary criterion: **F1** (ensures precision is not completely sacrificed)

Strict comparison across all 6 models without letting secondary narratives override metrics.

In [7]:
ranked = sorted(results, key=lambda x: (x['Recall'], x['F1']), reverse=True)
print('=== MODELS RANKED STRICTLY BY (RECALL desc, F1 desc) ===')
for i, r in enumerate(ranked, 1):
    print(f'{i}. {r["Model"]:35s} | Recall: {r["Recall"]:.4f} | F1: {r["F1"]:.4f} | Precision: {r["Precision"]:.4f} | ROC-AUC: {r["ROC-AUC"]:.4f}')

overall_winner = ranked[0]
print('\n' + '='*60)
print(f'HONEST WINNER: {overall_winner["Model"]}')
print('='*60)
print(f'Recall: {overall_winner["Recall"]:.4f}, F1: {overall_winner["F1"]:.4f}')

is_baseline = overall_winner['Model'] == 'Baseline LogReg (Scaled - Step 6)'
if is_baseline:
    print('RECOMMENDATION: Retain Baseline Logistic Regression (Scaled) as the production model.')
    print('It strictly dominates all unscaled and scaled tree models on both primary (Recall: 0.3617 vs max 0.2766) and secondary (F1: 0.4658 vs max 0.3714) criteria.')

=== MODELS RANKED STRICTLY BY (RECALL desc, F1 desc) ===
1. Baseline LogReg (Scaled - Step 6)   | Recall: 0.3617 | F1: 0.4658 | Precision: 0.6538 | ROC-AUC: 0.8134
2. XGBoost (Unscaled)                  | Recall: 0.2766 | F1: 0.3714 | Precision: 0.5652 | ROC-AUC: 0.7735
3. XGBoost (Scaled)                    | Recall: 0.2766 | F1: 0.3714 | Precision: 0.5652 | ROC-AUC: 0.7735
4. Logistic Regression (Unscaled)      | Recall: 0.2340 | F1: 0.3607 | Precision: 0.7857 | ROC-AUC: 0.7561
5. Random Forest (Unscaled)            | Recall: 0.1064 | F1: 0.1786 | Precision: 0.5556 | ROC-AUC: 0.7930
6. Random Forest (Scaled)              | Recall: 0.1064 | F1: 0.1786 | Precision: 0.5556 | ROC-AUC: 0.7913

HONEST WINNER: Baseline LogReg (Scaled - Step 6)
Recall: 0.3617, F1: 0.4658
RECOMMENDATION: Retain Baseline Logistic Regression (Scaled) as the production model.
It strictly dominates all unscaled and scaled tree models on both primary (Recall: 0.3617 vs max 0.2766) and secondary (F1: 0.4658 vs max 

---
## Step 6 · Root Cause Diagnosis: Tree Models vs. Class Imbalance

Why do tree ensembles underperform the baseline on Recall across both unscaled and scaled datasets?

In [8]:
diagnosis = '''
ROOT CAUSE DIAGNOSIS:
Evaluating Random Forest and XGBoost on scaled features confirmed that feature scaling is NOT 
the reason for their poor recall (decision tree splits are scale-invariant, yielding virtually identical 
or near-identical performance on scaled vs unscaled inputs). 

Instead, the primary driver is UNTREATED CLASS IMBALANCE in conjunction with the default 0.5 decision threshold:
1. In an 84/16 imbalanced dataset, standard Gini impurity / entropy in Random Forest and standard logistic loss 
   in XGBoost treat errors on both classes symmetrically. Since 84% of samples belong to the majority class (Stayed), 
   minimizing overall sample loss naturally drives tree leaf predictions toward the majority class.
2. Crucially, neither model in this default evaluation utilized class weighting — Random Forest was run without 
   `class_weight='balanced'` (or `'balanced_subsample'`), and XGBoost was run without setting `scale_pos_weight` 
   (which should ideally be ~(1233/237) ≈ 5.2 to penalize false negatives proportionally).
3. Applying a default 0.5 probability cutoff means an employee must have an overwhelming predicted probability 
   to be flagged, causing Random Forest to catch only 5 of 47 leavers and XGBoost only 13 of 47 leavers. 
   In contrast, L2-regularized Logistic Regression with scaled features produced better-spread linear log-odds 
   that managed to identify 17 of 47 leavers (Recall = 0.3617). Without explicit minority-class penalization or 
   threshold tuning, off-the-shelf tree ensembles severely penalize recall on imbalanced HR data.
'''
print(diagnosis.strip())

ROOT CAUSE DIAGNOSIS:
Evaluating Random Forest and XGBoost on scaled features confirmed that feature scaling is NOT 
the reason for their poor recall (decision tree splits are scale-invariant, yielding virtually identical 
or near-identical performance on scaled vs unscaled inputs). 

Instead, the primary driver is UNTREATED CLASS IMBALANCE in conjunction with the default 0.5 decision threshold:
1. In an 84/16 imbalanced dataset, standard Gini impurity / entropy in Random Forest and standard logistic loss 
   in XGBoost treat errors on both classes symmetrically. Since 84% of samples belong to the majority class (Stayed), 
   minimizing overall sample loss naturally drives tree leaf predictions toward the majority class.
2. Crucially, neither model in this default evaluation utilized class weighting — Random Forest was run without 
   `class_weight='balanced'` (or `'balanced_subsample'`), and XGBoost was run without setting `scale_pos_weight` 
   (which should ideally be ~(1233/237) ≈ 

---
## Step 7 · Save Production Pipeline

Saving the best-performing model (`Baseline LogReg Scaled`, or best candidate per rule) to `models/attrition_pipeline.joblib`.

In [9]:
# If Baseline LogReg is the honest winner, we save the trained baseline model to models/attrition_pipeline.joblib
out_file = os.path.join(MODELS, 'attrition_pipeline.joblib')

if overall_winner['Model'] == 'Baseline LogReg (Scaled - Step 6)':
    baseline_source = os.path.join(MODELS, 'baseline_logreg.joblib')
    if os.path.exists(baseline_source):
        winner_obj = joblib.load(baseline_source)
    else:
        # Retrain baseline model
        winner_obj = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        winner_obj.fit(X_train_s, y_train_s)
    joblib.dump(winner_obj, out_file)
    print(f'Saved Baseline Logistic Regression (Scaled) to: {out_file}')
else:
    # If another model won, save that
    winner_name = overall_winner['Model']
    # find model obj
    m_obj = models_unscaled.get(winner_name) or models_scaled.get(winner_name)
    joblib.dump(m_obj, out_file)
    print(f'Saved {winner_name} to: {out_file}')

print(f'File size: {os.path.getsize(out_file):,} bytes')
loaded = joblib.load(out_file)
print(f'Verified pipeline type: {type(loaded)}')

# Also print feature importances / coefficients for the retained model
if hasattr(loaded, 'coef_'):
    coef_s = pd.Series(loaded.coef_[0], index=X_scaled.columns).abs().sort_values(ascending=False).head(15)
    print('\nTop 15 Absolute Coefficients of production model:')
    print(coef_s.to_string())
elif hasattr(loaded, 'feature_importances_'):
    fi_s = pd.Series(loaded.feature_importances_, index=X_unscaled.columns).sort_values(ascending=False).head(15)
    print('\nTop 15 Feature Importances of production model:')
    print(fi_s.to_string())

Saved Baseline Logistic Regression (Scaled) to: ..\models\attrition_pipeline.joblib
File size: 2,575 bytes
Verified pipeline type: <class 'sklearn.linear_model._logistic.LogisticRegression'>

Top 15 Absolute Coefficients of production model:
OverTime                            1.8021
BusinessTravel_Travel_Frequently    1.5569
JobRole_Laboratory Technician       1.3029
JobRole_Sales Representative        0.9187
EducationField_Other                0.8919
YearsSinceLastPromotion             0.8416
JobRole_Research Director           0.8213
TotalWorkingYears                   0.6971
MaritalStatus_Single                0.6965
BusinessTravel_Travel_Rarely        0.6689
Department_Research & Development   0.5799
JobRole_Human Resources             0.5133
NumCompaniesWorked                  0.4284
EducationField_Medical              0.4170
Age                                 0.3997
